# 02 — هم‌ترازی Outcomeهای اجتماعی و تغییرهای مالی

این Notebook فقط پس از نهایی‌شدن Eligibility و برچسب‌های X، Reddit و YouTube اجرا می‌شود. نبود ورودی اجتماعی در این مرحله یک وضعیت برنامه‌ریزی‌شده است.

## قرارداد آماری

- Outcome اجتماعی هفته `t` با تغییر مالی هفته `t + lag` مقایسه می‌شود.
- Lagهای ثبت‌شده: `0`, `1`, `2` هفته.
- روش اصلی: Spearman با `p-value` حاصل از ۹٬۹۹۹ Permutation جفت‌شده.
- فاصله اطمینان: Bootstrap درصدی جفت‌های هفته.
- حساسیت‌ها: Pearson، حذف W21 پنج‌روزه و دو شاخص جایگزین ثبت‌شده.
- اصلاح چندگانگی: Benjamini–Hochberg FDR.
- حداقل تحلیل: ۱۰ هفته جفت‌شده و دو سری غیرثابت.
- تفسیر: Association زمانی، نه رابطه علّی.

In [1]:
from pathlib import Path

def find_project_root(start=None):
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "config" / "config.yaml").exists() and (candidate / "data" / "interim" / "financial" / "frozen_inputs").exists():
            return candidate
    raise FileNotFoundError("Project root with frozen financial inputs was not found.")

ROOT = find_project_root()
ROOT

WindowsPath('C:/Users/user/OneDrive/Desktop/hamrahaval/final/media-sentiment-pipeline-starter/media-sentiment-pipeline-starter')

## ۱. بارگذاری ورودی مالی و بررسی ورودی اجتماعی

In [2]:
import sys

# The analysis needs pandas/numpy only; optional binary add-ons are not required.
for optional_module in ["pyarrow", "numexpr", "bottleneck"]:
    sys.modules.setdefault(optional_module, None)

import numpy as np
import pandas as pd

WEEKLY_PATH = ROOT / "outputs" / "tables" / "financial" / "financial_weekly_returns_v1.csv"
SOCIAL_PATH = ROOT / "data" / "processed" / "social_media" / "social_weekly_outcomes_v1.csv"
RESULT_PATH = ROOT / "outputs" / "tables" / "financial" / "financial_social_correlation_results_v1.csv"

weekly = pd.read_csv(WEEKLY_PATH)
social_ready = SOCIAL_PATH.exists() and SOCIAL_PATH.stat().st_size > 0
print({"financial_rows": len(weekly), "social_input_exists": SOCIAL_PATH.exists(), "social_ready": social_ready})

{'financial_rows': 157, 'social_input_exists': True, 'social_ready': True}


## ۲. توابع آماری شفاف

In [3]:
SEED = 1405
N_PERMUTATIONS = 9999
N_BOOTSTRAP = 2000
MIN_PAIRED_WEEKS = 10

def correlation(x, y, method):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if method == "spearman":
        x = pd.Series(x).rank(method="average").to_numpy()
        y = pd.Series(y).rank(method="average").to_numpy()
    return float(np.corrcoef(x, y)[0, 1])

def permutation_pvalue(x, y, observed, method, rng, n_perm=N_PERMUTATIONS):
    exceed = 0
    for _ in range(n_perm):
        permuted = rng.permutation(y)
        if abs(correlation(x, permuted, method)) >= abs(observed):
            exceed += 1
    return (exceed + 1) / (n_perm + 1)

def bootstrap_ci(x, y, method, rng, n_boot=N_BOOTSTRAP):
    n = len(x)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        xb, yb = np.asarray(x)[idx], np.asarray(y)[idx]
        if np.std(xb) > 0 and np.std(yb) > 0:
            values.append(correlation(xb, yb, method))
    if not values:
        return np.nan, np.nan
    return tuple(np.quantile(values, [0.025, 0.975]))

def bh_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = np.minimum.accumulate((ranked * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.minimum(adjusted_ranked, 1.0)
    return adjusted

## ۳. ساخت جفت‌های هفته و اجرای تحلیل اصلی و حساسیت‌ها

In [4]:
REQUIRED_SOCIAL_COLUMNS = {
    "platform", "project_week", "outcome_id", "target_id", "outcome_value", "n_records"
}

ANALYSIS_VARIANTS = [
    {"variant": "primary_all_weeks", "asset_use": "main", "methods": ["spearman", "pearson"], "exclude_w21": False},
    {"variant": "sensitivity_exclude_w21", "asset_use": "main", "methods": ["spearman"], "exclude_w21": True},
    {"variant": "sensitivity_alternative_assets", "asset_use": "sensitivity", "methods": ["spearman"], "exclude_w21": False},
]

def run_alignment(weekly, social):
    missing = REQUIRED_SOCIAL_COLUMNS - set(social.columns)
    if missing:
        raise ValueError(f"Missing social columns: {sorted(missing)}")
    social = social.copy()
    social["week_number"] = social["project_week"].str[1:].astype(int)
    financial = weekly.loc[weekly["analysis_use"].isin(["main", "sensitivity"])].copy()
    financial["financial_week_number"] = financial["project_week"].str[1:].astype(int)
    rows = []
    for variant in ANALYSIS_VARIANTS:
        variant_financial = financial.loc[financial["analysis_use"] == variant["asset_use"]]
        for social_key, s in social.groupby(["platform", "outcome_id", "target_id"], dropna=False):
            for asset_id, f in variant_financial.groupby("asset_id"):
                for lag in [0, 1, 2]:
                    shifted = f[["financial_week_number", "weekly_simple_change"]].copy()
                    shifted["social_week_number"] = shifted["financial_week_number"] - lag
                    paired = s.merge(shifted, left_on="week_number", right_on="social_week_number", how="inner")
                    if variant["exclude_w21"]:
                        paired = paired.loc[(paired["week_number"] != 21) & (paired["financial_week_number"] != 21)]
                    paired = paired.dropna(subset=["outcome_value", "weekly_simple_change"])
                    x = paired["outcome_value"].to_numpy(float)
                    y = paired["weekly_simple_change"].to_numpy(float)
                    for method in variant["methods"]:
                        family = ("primary_spearman" if variant["variant"] == "primary_all_weeks" and method == "spearman"
                                  else f"{variant['variant']}_{method}")
                        base = {
                            "platform": social_key[0], "outcome_id": social_key[1], "target_id": social_key[2],
                            "analysis_variant": variant["variant"], "test_family": family,
                            "asset_id": asset_id, "asset_analysis_use": variant["asset_use"],
                            "financial_lag_weeks": lag, "n_paired_weeks": len(paired), "method": method,
                        }
                        if len(paired) < MIN_PAIRED_WEEKS or np.std(x) == 0 or np.std(y) == 0:
                            rows.append({**base, "coefficient": np.nan, "ci_low": np.nan, "ci_high": np.nan,
                                         "p_value_raw": np.nan, "status": "insufficient_or_constant"})
                            continue
                        seed_offset = sum(ord(ch) for ch in f"{social_key}-{asset_id}-{lag}-{method}-{variant['variant']}")
                        rng = np.random.default_rng(SEED + seed_offset)
                        coefficient = correlation(x, y, method)
                        p_raw = permutation_pvalue(x, y, coefficient, method, rng)
                        ci_low, ci_high = bootstrap_ci(x, y, method, rng)
                        rows.append({**base, "coefficient": coefficient, "ci_low": ci_low, "ci_high": ci_high,
                                     "p_value_raw": p_raw, "status": "exploratory_association_not_causal"})
    result = pd.DataFrame(rows)
    result["p_value_bh_fdr"] = np.nan
    eligible = result.loc[result["p_value_raw"].notna()]
    for family, idx in eligible.groupby("test_family").groups.items():
        result.loc[idx, "p_value_bh_fdr"] = bh_adjust(result.loc[idx, "p_value_raw"])
    return result

## ۴. اجرای مشروط و ثبت وضعیت

In [5]:
if not social_ready:
    print("STATUS: pending_social_outcomes")
    print(f"Place the reviewed weekly outcome file at: {SOCIAL_PATH}")
    results = pd.DataFrame()
else:
    social = pd.read_csv(SOCIAL_PATH)
    results = run_alignment(weekly, social)
    RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(RESULT_PATH, index=False)
    print(f"Saved {len(results):,} registered tests to {RESULT_PATH}")

results.head()

Saved 756 registered tests to C:\Users\user\OneDrive\Desktop\hamrahaval\final\media-sentiment-pipeline-starter\media-sentiment-pipeline-starter\outputs\tables\financial\financial_social_correlation_results_v1.csv


,platform,outcome_id,target_id,analysis_variant,test_family,asset_id,asset_analysis_use,financial_lag_weeks,n_paired_weeks,method,coefficient,ci_low,ci_high,p_value_raw,status,p_value_bh_fdr
0,reddit,anger_share,NaN,primary_all_weeks,primary_spearman,GOLD_USD,main,0,21,spearman,-0.041585,-0.529292,0.424789,0.8576,exploratory_association_not_causal,0.98490
1,reddit,anger_share,NaN,primary_all_weeks,primary_all_weeks_pearson,GOLD_USD,main,0,21,pearson,-0.149037,-0.561092,0.403329,0.5189,exploratory_association_not_causal,0.98670
2,reddit,anger_share,NaN,primary_all_weeks,primary_spearman,GOLD_USD,main,1,20,spearman,-0.020316,-0.495624,0.408438,0.9344,exploratory_association_not_causal,0.98490
3,reddit,anger_share,NaN,primary_all_weeks,primary_all_weeks_pearson,GOLD_USD,main,1,20,pearson,-0.114841,-0.561657,0.429981,0.6288,exploratory_association_not_causal,0.98670
4,reddit,anger_share,NaN,primary_all_weeks,primary_spearman,GOLD_USD,main,2,19,spearman,0.411765,-0.135087,0.786828,0.0824,exploratory_association_not_causal,0.85608


## قاعده گزارش

ضریب و فاصله اطمینان از `p-value` مهم‌ترند. نتیجه با `p_value_bh_fdr < 0.05` فقط به‌عنوان شواهد ارتباط زمانی در نمونه مشاهده‌شده گزارش می‌شود. نبود نتیجه معنادار نیز اثبات نبود رابطه نیست، به‌ویژه با حداکثر ۲۱ هفته.